# 02 · Validity checks

**Goal:** before estimating any effect, check that the experiment itself can be trusted.

1. **Sample ratio mismatch (SRM):** do the observed arm sizes match the designed 1/3-1/3-1/3 split?
2. **Covariate balance:** do pre-treatment customer characteristics look the same across arms?

If either check failed, the effect estimates in notebook 03 could be biased no matter how carefully they are computed.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hillstrom.data import load_customers
from hillstrom.plots import ARM_COLORS, ARM_LABELS, TEXT_SECONDARY, apply_style, save_figure
from hillstrom.validity import balance_table, srm_test

pd.set_option("display.float_format", "{:,.4f}".format)
apply_style()

df = load_customers()

## 1. Sample ratio mismatch

An SRM means the split that happened is not the split that was designed, which usually points to a bug in assignment or data logging (for example, some customers in one arm dropped out of the data). When that happens, the arms are no longer comparable and the experiment should not be analysed as-is.

**Test:** chi-square goodness-of-fit of the three arm counts against equal thirds (2 degrees of freedom). One test covers all three arms at once; pairwise binomial tests would need three tests and a multiple-comparison fix.

**Threshold:** SRM checks are usually flagged at p < 0.001 rather than 0.05. The check is run routinely on every experiment, so a 0.05 threshold would raise a false alarm on 1 in 20 healthy experiments, while a real assignment bug typically produces a far smaller p-value.

In [2]:
counts = df["arm"].value_counts().sort_index()
srm = srm_test(counts.to_numpy(), [1 / 3, 1 / 3, 1 / 3])

pd.DataFrame(
    {"observed": srm.counts, "expected": srm.expected, "observed_share": srm.counts / srm.counts.sum()},
    index=counts.index,
).style.format({"observed": "{:,.0f}", "expected": "{:,.1f}", "observed_share": "{:.4f}"})

,observed,expected,observed_share
arm,,,
control,"21,306","21,333.3",0.3329
mens,"21,307","21,333.3",0.3329
womens,"21,387","21,333.3",0.3342


In [3]:
print(f"chi2 = {srm.chi2:.3f}, p = {srm.p_value:.3f}")

chi2 = 0.203, p = 0.904


**Result:** the largest deviation from the expected count is about 50 customers out of ~21,300 per arm, and p is far above 0.001. There is no sign of a sample ratio mismatch.

What this does *not* show: SRM only checks group sizes. A randomization bug that kept sizes equal but sorted customers into arms by some characteristic would pass this test. The balance check below covers that case.